# 03 Silver Patient Clean

## Purpose

This notebook creates the Silver Patient table from the Bronze raw Patient FHIR table.

## What We Are Doing

We will:
1. Read `healthcare_catalog.bronze.patient_raw`
2. Extract useful patient fields from nested FHIR JSON
3. Flatten patient demographics
4. Create a clean patient table
5. Save it as `healthcare_catalog.silver.patient_clean`

## Why We Are Doing This

Bronze data preserves the raw FHIR structure.

Silver data makes the data easier to use for:
- SQL analytics
- dashboards
- machine learning
- patient risk modeling
- population health analytics

## Expected Final Output

A clean Silver table:

`healthcare_catalog.silver.patient_clean`

Expected columns:
- patient_id
- full_url
- gender
- birth_date
- city
- state
- country
- postal_code
- family_name
- given_name
- marital_status

## Step 1 — Import PySpark Functions

### What We Are Doing

We are importing PySpark SQL functions.

### Why We Are Doing This

We need Spark functions to select nested fields, rename columns, and transform FHIR data.

### Expected Output

Spark functions will be available for this notebook.

In [0]:
from pyspark.sql.functions import *

## Step 2 — Read Bronze Patient Table

### What We Are Doing

We are reading the raw Patient Bronze Delta table.

### Why We Are Doing This

The Bronze table contains raw Patient FHIR resources.  
We need this as the input for Silver cleaning.

### Expected Output

A DataFrame named `patient_raw_df`.

In [0]:
patient_raw_df = spark.table("healthcare_catalog.bronze.patient_raw")

print("Bronze patient_raw table loaded successfully.")

Bronze patient_raw table loaded successfully.


## Step 3 — Inspect Patient Raw Schema

### What We Are Doing

We are printing the schema of the raw Patient table.

### Why We Are Doing This

FHIR Patient data is nested.  
Before extracting fields, we need to understand the structure.

### Expected Output

You should see:
- fullUrl
- resourceType
- resource
- nested fields inside resource such as id, gender, birthDate, address, name, maritalStatus

In [0]:
patient_raw_df.printSchema()

root
 |-- fullUrl: string (nullable = true)
 |-- resourceType: string (nullable = true)
 |-- resource: struct (nullable = true)
 |    |-- abatementDateTime: string (nullable = true)
 |    |-- active: boolean (nullable = true)
 |    |-- activity: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- detail: struct (nullable = true)
 |    |    |    |    |-- code: struct (nullable = true)
 |    |    |    |    |    |-- coding: array (nullable = true)
 |    |    |    |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |    |    |    |-- system: string (nullable = true)
 |    |    |    |    |    |-- text: string (nullable = true)
 |    |    |    |    |-- location: struct (nullable = true)
 |    |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- statu

## Step 4 — Extract Clean Patient Columns

### What We Are Doing

We are extracting useful patient demographic fields from the nested FHIR Patient resource.

### Why We Are Doing This

FHIR Patient resources are deeply nested JSON structures.

Analytics and machine learning require flat tabular columns.

We are converting nested FHIR Patient data into a clean Silver patient table.

### Fields We Will Extract

- patient_id
- full_url
- gender
- birth_date
- city
- state
- country
- postal_code
- family_name
- given_name
- marital_status

### Expected Output

A clean DataFrame named:

`patient_clean_df`

Each row will represent one patient with flattened demographic information.

In [0]:
patient_clean_df = patient_raw_df.select(

    col("resource.id").alias("patient_id"),

    col("fullUrl").alias("full_url"),

    col("resource.gender").alias("gender"),

    col("resource.birthDate").alias("birth_date"),

    get_json_object(col("resource.address"), "$[0].city").alias("city"),

    get_json_object(col("resource.address"), "$[0].state").alias("state"),

    get_json_object(col("resource.address"), "$[0].country").alias("country"),

    get_json_object(col("resource.address"), "$[0].postalCode").alias("postal_code"),

    get_json_object(col("resource.name"), "$[0].family").alias("family_name"),

    get_json_object(col("resource.name"), "$[0].given[0]").alias("given_name"),

    col("resource.maritalStatus.text").alias("marital_status")
)

print("Patient clean DataFrame created successfully.")

Patient clean DataFrame created successfully.


## Step 5 — Inspect Clean Patient Data

### What We Are Doing

We are displaying the flattened patient demographic data.

### Why We Are Doing This

We need to verify:
- nested extraction worked correctly
- columns are flattened properly
- demographic values appear correctly

### Expected Output

You should see clean columns such as:
- patient_id
- gender
- city
- state
- birth_date
- family_name

In [0]:
display(patient_clean_df)

patient_id,full_url,gender,birth_date,city,state,country,postal_code,family_name,given_name,marital_status
d7c46304-29f5-5ddb-f5df-7d816cf4f318,urn:uuid:d7c46304-29f5-5ddb-f5df-7d816cf4f318,female,1983-08-02,Westwood,MA,US,null,Herman763,Elana101,M
988ba5c5-bb2c-1453-cfe7-16ea41c47b42,urn:uuid:988ba5c5-bb2c-1453-cfe7-16ea41c47b42,female,1980-03-25,Acushnet,MA,US,null,Hyatt152,Eleanora667,M
246fb368-8991-dc93-f6a5-eca807e7dbde,urn:uuid:246fb368-8991-dc93-f6a5-eca807e7dbde,female,1990-10-28,Halifax,MA,US,null,Hilpert278,Eleni953,S
8daa23b6-137c-abd3-58c8-da0f70b98143,urn:uuid:8daa23b6-137c-abd3-58c8-da0f70b98143,female,2015-03-31,Plymouth,MA,US,null,Buckridge80,Elenora790,Never Married
7d5e31d3-163b-4a77-576b-1ed21adf8c09,urn:uuid:7d5e31d3-163b-4a77-576b-1ed21adf8c09,male,2021-01-24,Springfield,MA,US,01105,Douglas31,Eli762,Never Married
5c15f3b9-fbd5-80e5-755d-5a697e4b0499,urn:uuid:5c15f3b9-fbd5-80e5-755d-5a697e4b0499,female,2003-02-28,Peabody,MA,US,01940,Windler79,Ellen406,Never Married
03777c32-ed98-50e2-f75d-cbcad532c610,urn:uuid:03777c32-ed98-50e2-f75d-cbcad532c610,female,1968-03-26,Sudbury,MA,US,null,Gerhold939,Eloise59,M
22b01ffb-7537-e873-3a7c-0d6961c498d7,urn:uuid:22b01ffb-7537-e873-3a7c-0d6961c498d7,male,2018-10-29,Middleborough,MA,US,null,Homenick806,Elton404,Never Married
53cc5b94-3c84-3ecf-ae94-f98203e3d8ba,urn:uuid:53cc5b94-3c84-3ecf-ae94-f98203e3d8ba,male,2014-03-07,Braintree,MA,US,02184,Gottlieb798,Elwood28,Never Married
e65573ba-e39e-bac4-2e49-2d3cf9a538b8,urn:uuid:e65573ba-e39e-bac4-2e49-2d3cf9a538b8,female,1965-06-03,Braintree,MA,US,02184,Johnson679,Elza246,S


## Step 6 — Check Null Values

### What We Are Doing

We are checking whether important demographic columns contain null values.

### Why We Are Doing This

Healthcare data often contains:
- missing demographics
- incomplete addresses
- inconsistent fields

Data quality validation is a critical part of Silver layer engineering.

### Expected Output

Null counts for important columns.

In [0]:
display(
    patient_clean_df.select(
        [
            sum(col(column_name).isNull().cast("int")).alias(column_name)
            for column_name in patient_clean_df.columns
        ]
    )
)

patient_id,full_url,gender,birth_date,city,state,country,postal_code,family_name,given_name,marital_status
0,0,0,0,0,0,0,267,0,0,0


## Step 7 — Save Silver Patient Table

### What We Are Doing

We are saving the clean patient demographic table into the Silver layer.

### Why We Are Doing This

The Silver layer stores:
- cleaned
- flattened
- analytics-ready

healthcare tables.

This table will later support:
- dashboards
- ML feature engineering
- population health analytics
- patient risk models

### Expected Output

A Delta table:

`healthcare_catalog.silver.patient_clean`

In [0]:
patient_clean_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("healthcare_catalog.silver.patient_clean")

print("Silver patient_clean table saved successfully.")

Silver patient_clean table saved successfully.


## Step 8 — Verify Silver Patient Table

### What We Are Doing

We are verifying that the Silver patient table was successfully created.

### Why We Are Doing This

Verification ensures:
- Delta save succeeded
- table is queryable
- schema exists correctly

### Expected Output

You should see:
- patient_clean
inside:
- healthcare_catalog.silver

In [0]:
spark.sql("""
SHOW TABLES IN healthcare_catalog.silver
""").show(truncate=False)

+--------+-------------+-----------+
|database|tableName    |isTemporary|
+--------+-------------+-----------+
|silver  |patient_clean|false      |
+--------+-------------+-----------+

